LOUISE MARIELLE V. DORADO
STEVEN KEN E. PONTILLAS
BSCS 3A AI

Exercise for Unit 4.1 Naïve Bayes

1. Performing it manually.

In [9]:
#a. Generate a Bag of Words (for word frequency)

from collections import Counter
import re

# Dataset from the prompt
DOCS = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM"),
]


def tokenize(text: str) -> list[str]:
    """Lowercase and keep numbers; keep internal apostrophes (e.g., let's)."""
    return re.findall(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?", text.lower())


def bag_of_words_per_class(docs: list[tuple[str, str]]) -> tuple[set[str], dict[str, Counter]]:
    """Return overall vocabulary and per-class word frequency counts."""
    vocab: set[str] = set()
    counts_by_class: dict[str, Counter] = {}

    for text, label in docs:
        tokens = tokenize(text)
        vocab.update(tokens)
        if label not in counts_by_class:
            counts_by_class[label] = Counter()
        counts_by_class[label].update(tokens)

    return vocab, counts_by_class


vocab, counts_by_class = bag_of_words_per_class(DOCS)

vocab, counts_by_class

({'3',
  '50',
  'a',
  'are',
  'at',
  'can',
  'catch',
  'click',
  'dinner',
  'for',
  'free',
  'get',
  'here',
  'hi',
  'how',
  'in',
  'iphone',
  "let's",
  'limited',
  'lowest',
  'meds',
  'meeting',
  'mom',
  'money',
  'now',
  'off',
  'office',
  'on',
  'pm',
  'price',
  'prizes',
  'report',
  'send',
  'still',
  'team',
  'the',
  'time',
  'today',
  'tomorrow',
  'up',
  'we',
  'win',
  'you',
  'your'},
 {'SPAM': Counter({'free': 2,
           'for': 2,
           'money': 1,
           'now': 1,
           'lowest': 1,
           'price': 1,
           'your': 1,
           'meds': 1,
           'win': 1,
           'a': 1,
           'iphone': 1,
           'today': 1,
           'get': 1,
           '50': 1,
           'off': 1,
           'limited': 1,
           'time': 1,
           'click': 1,
           'here': 1,
           'prizes': 1}),
  'HAM': Counter({'the': 3,
           'are': 2,
           'you': 2,
           'tomorrow': 2,
           'at

In [10]:
# b. Calculate the prior for the class HAM and SPAM
num_docs = len(DOCS)
class_counts = {label: 0 for label in counts_by_class}

for _, label in DOCS:
    class_counts[label] += 1

priors = {label: class_counts[label] / num_docs for label in class_counts}

class_counts, priors

({'SPAM': 5, 'HAM': 6},
 {'SPAM': 0.45454545454545453, 'HAM': 0.5454545454545454})

In [11]:
# c. Calculate the likelihood of the tokens in the vocabulary with respect to the class.
vocab_size = len(vocab)

likelihoods = {}
for label, token_counts in counts_by_class.items():
    total_tokens = sum(token_counts.values())
    denom = total_tokens + vocab_size
    likelihoods[label] = {token: (token_counts.get(token, 0) + 1) / denom for token in vocab}

# Show a small sample for each class
sample_tokens = sorted(vocab)[:10]
{label: {t: likelihoods[label][t] for t in sample_tokens} for label in likelihoods}

{'SPAM': {'3': 0.015151515151515152,
  '50': 0.030303030303030304,
  'a': 0.030303030303030304,
  'are': 0.015151515151515152,
  'at': 0.015151515151515152,
  'can': 0.015151515151515152,
  'catch': 0.015151515151515152,
  'click': 0.030303030303030304,
  'dinner': 0.015151515151515152,
  'for': 0.045454545454545456},
 'HAM': {'3': 0.025974025974025976,
  '50': 0.012987012987012988,
  'a': 0.012987012987012988,
  'are': 0.03896103896103896,
  'at': 0.03896103896103896,
  'can': 0.025974025974025976,
  'catch': 0.025974025974025976,
  'click': 0.012987012987012988,
  'dinner': 0.025974025974025976,
  'for': 0.025974025974025976}}

In [12]:
# d. Determine the class of the following test sentence:

# Ensure required values exist if prior cells were not run
if "counts_by_class" not in globals() or "vocab" not in globals():
    vocab, counts_by_class = bag_of_words_per_class(DOCS)

if "priors" not in globals():
    num_docs = len(DOCS)
    class_counts = {label: 0 for label in counts_by_class}
    for _, label in DOCS:
        class_counts[label] += 1
    priors = {label: class_counts[label] / num_docs for label in class_counts}

vocab_size = len(vocab)

# Classify new sentences using Naive Bayes (plain probabilities)
class_token_totals = {label: sum(token_counts.values()) for label, token_counts in counts_by_class.items()}
class_denoms = {label: class_token_totals[label] + vocab_size for label in counts_by_class}


def predict_class(text: str) -> tuple[str, dict[str, float]]:
    tokens = tokenize(text)
    scores: dict[str, float] = {}
    for label in priors:
        prob = priors[label]
        for token in tokens:
            if token in vocab:
                prob *= likelihoods[label][token]
            else:
                prob *= 1 / class_denoms[label]
        scores[label] = prob
    predicted = max(scores, key=scores.get)
    return predicted, scores


test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager.",
]

[(s, *predict_class(s)) for s in test_sentences]

[('Limited offer, click here!',
  'SPAM',
  {'SPAM': 1.9164238366023312e-07, 'HAM': 1.551656783987893e-08}),
 ('Meeting at 2 PM with the manager.',
  'HAM',
  {'SPAM': 8.332393479397675e-14, 'HAM': 2.4471240512104998e-12})]

2. Using Scikit-Learn.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Train Multinomial Naive Bayes with scikit-learn
texts = [text for text, _ in DOCS]
labels = [label for _, label in DOCS]

vectorizer = CountVectorizer(lowercase=True, token_pattern=r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")
X = vectorizer.fit_transform(texts)

clf = MultinomialNB(alpha=1.0)
clf.fit(X, labels)

sk_test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager.",
]
X_test = vectorizer.transform(sk_test_sentences)

list(zip(sk_test_sentences, clf.predict(X_test)))

[('Limited offer, click here!', 'SPAM'),
 ('Meeting at 2 PM with the manager.', 'HAM')]